# Iniciando o Spark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Trusted_base_atraso").getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import sys
import pytz
import numpy as np
import datetime
from pyspark.sql import SQLContext
#from pyspark.sql.functions import udf, lpad, translate
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
#from pyspark.sql.types import *
from pyspark.sql.functions import count, avg

#Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

# Buckets e nomes de saída
bucket_base = "base_atraso"
bucket_trusted = f"s3://meu-bucket/{PROCESS_DATE}/0003_trusted/{bucket_base}"
bucket_raw = f"s3://meu-bucket/{PROCESS_DATE}/0002_raw/{bucket_base}"
bucket_control = f"s3://meu-bucket/{PROCESS_DATE}/0005_control/{bucket_base}"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


#  Leitura da camada Raw

In [ ]:
#raw_path = os.path.join(bucket_raw, "base_atraso")

path_raw = os.path.join(bucket_raw, bucket_base)

parquet_files = [path_raw for f in os.listdir(bucket_raw) if f.endswith('.parquet')]
df_raw_atraso= spark.read.parquet(*parquet_files, header=True, inferSchema=True)

df_raw_atraso.createOrReplaceTempView("raw_base_atraso")

print(log(), "Registros na Raw:", df_raw_atraso.count())
df_raw_atraso.show(5, truncate=False)

# Processamento tipagem para camada Trusted

In [ ]:
df_trusted_atraso = spark.sql("""
SELECT
'{dt_proc}' as DATA_IMPORT,
    CAST(NUM_CPF AS STRING)                                     AS NUM_CPF,
    TO_DATE(DAT_REFERENCIA, 'ddMMMyyyy:HH:mm:ss')               AS DAT_REFERENCIA,
    CAST(NUM_FATURA_HASH AS STRING)                              AS NUM_FATURA_HASH,
    CAST(NUM_ENT_SEQ_FATURA AS INT)                              AS NUM_ENT_SEQ_FATURA,
    CAST(CONTRATO AS BIGINT)                                     AS CONTRATO,
    CAST(DW_UN_NEGOCIO AS INT)                                   AS DW_UN_NEGOCIO,
    CAST(DW_HIS_PONTO_VENDA_COMTA AS BIGINT)                     AS DW_HIS_PONTO_VENDA_COMTA,
    CAST(DW_NUM_CLIENTE AS BIGINT)                               AS DW_NUM_CLIENTE,
    CAST(DW_AREA AS INT)                                         AS DW_AREA,
    CAST(DW_CICLO AS INT)                                        AS DW_CICLO,
    CAST(DW_TIPO_CLIENTE_CONTA AS INT)                           AS DW_TIPO_CLIENTE_CONTA,
    CAST(DW_OFERTA AS INT)                                       AS DW_OFERTA,
    CAST(DW_FAIXA_AGING_FATURA AS INT)                           AS DW_FAIXA_AGING_FATURA,
    CAST(DW_FAIXA_AGING_DIVIDA AS INT)                           AS DW_FAIXA_AGING_DIVIDA,
    CAST(DW_FAIXA_TEMPO_BASE AS INT)                             AS DW_FAIXA_TEMPO_BASE,
    CAST(DW_FAIXA_AGING_PROX_FECH AS INT)                        AS DW_FAIXA_AGING_PROX_FECH,
    CAST(DW_TIPO_FATURAMENTO AS INT)                             AS DW_TIPO_FATURAMENTO,
    CAST(COD_PLATAFORMA AS STRING)                               AS COD_PLATAFORMA,

    TO_DATE(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')    AS DAT_CRIACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_CRIACAO_REGISTRO_TRANS,
    TO_DATE(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')                           AS DAT_ALTERACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')  AS HR_ALTERACAO_REGISTRO_TRANS,
    TO_DATE(DAT_CANCELAMENTO_FAT, 'ddMMMyyyy:HH:mm:ss')         AS DAT_CANCELAMENTO_FAT,
    TO_DATE(DAT_ORIGINAL_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')         AS DAT_ORIGINAL_VCTO_FAT,
    TO_DATE(DAT_ALTERACAO_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')        AS DAT_ALTERACAO_VCTO_FAT,
    TO_DATE(DAT_CRIACAO_FAT,'ddMMMyyyy:HH:mm:ss')               AS DAT_CRIACAO_FAT,
    TO_DATE(DAT_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')            AS DAT_VENCIMENTO_FAT,
    TO_DATE(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss')                AS DAT_STATUS_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')                AS HR_STATUS_FAT,
    TO_DATE(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')                                 AS DAT_MIN_VENCIMENTO_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')        AS HR_MIN_VENCIMENTO_FAT,

    CAST(NUM_BILL_SEQ_FAT AS INT)                                AS NUM_BILL_SEQ_FAT,
    CAST(NUM_SEQ_ACORDO_FAT AS INT)                              AS NUM_SEQ_ACORDO_FAT,

    CAST(IND_ISENCAO_COB_FAT AS STRING)                          AS IND_ISENCAO_COB_FAT,
    CAST(IND_WO AS STRING)                                       AS IND_WO,
    CAST(IND_PDD AS STRING)                                      AS IND_PDD,
    CAST(IND_PCCR AS STRING)                                     AS IND_PCCR,
    CAST(IND_ACA AS STRING)                                      AS IND_ACA,
    CAST(IND_PRIMEIRA_FAT AS STRING)                             AS IND_PRIMEIRA_FAT,
    CAST(IND_FRAUDE AS STRING)                                   AS IND_FRAUDE,

    CAST(VAL_FAT_LIQUIDO AS DECIMAL(18,2))                       AS VAL_FAT_LIQUIDO,
    CAST(VAL_FAT_BRUTO AS DECIMAL(18,2))                         AS VAL_FAT_BRUTO,
    CAST(VAL_FAT_CREDITO AS DECIMAL(18,2))                       AS VAL_FAT_CREDITO,
    CAST(VAL_FAT_AJUSTE AS DECIMAL(18,2))                        AS VAL_FAT_AJUSTE,
    CAST(VAL_FAT_BRUTO_BC AS DECIMAL(18,2))                      AS VAL_FAT_BRUTO_BC,
    CAST(VAL_FAT_PAGAMENTO_BRUTO AS DECIMAL(18,2))               AS VAL_FAT_PAGAMENTO_BRUTO,
    CAST(VAL_FAT_ABERTO AS DECIMAL(18,2))                        AS VAL_FAT_ABERTO,
    CAST(VAL_FAT_ABERTO_LIQ AS DECIMAL(18,2))                    AS VAL_FAT_ABERTO_LIQ,
    CAST(VAL_MULTA_JUROS AS DECIMAL(18,2))                       AS VAL_MULTA_JUROS,
    CAST(VAL_MULTA_CANCELAMENTO AS DECIMAL(18,2))               AS VAL_MULTA_CANCELAMENTO,
    CAST(VAL_PARC_APARELHO_LIQ AS DECIMAL(18,2))                 AS VAL_PARC_APARELHO_LIQ,
    CAST(VAL_FAT_LIQ_JM_MC AS DECIMAL(18,2))                     AS VAL_FAT_LIQ_JM_MC,

    TO_DATE(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss')    AS DAT_ATIVACAO_CONTA_CLI,
    TO_CHAR(TO_TIMESTAMP(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_ATIVACAO_CONTA_CLI,
    TO_DATE(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss')            AS DAT_CRIACAO_DW,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')            AS HR_CRIACAO_DW
FROM raw_base_atraso
""")
df_trusted_atraso.createOrReplaceTempView("trusted_base_atraso")
df_trusted_atraso.cache()

print(log(), "Registros Trusted:", df_trusted_atraso.count())
df_trusted_atraso.printSchema()
#df_trusted_atraso.show(5, truncate=False)

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
df_trusted_atraso.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
df_controle = spark.sql (f"""
SELECT
'{dthproc}' as DATA_IMPORT,
count(*) as qtd_registros
from trusted_base_atraso
""")

df_controle.show()

# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}')
print("Control path:", path_control)

df_controle.write \
  .mode('append') \
  .option('compression', 'snappy') \
  .parquet(path_control)